<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: The errors that matter

**[The errors that matter](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/)** · Speech AI: Transcribe, Understand and Speak · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** count the errors on the words that matter to Larkfield (order numbers, product names, dates and negations) apart from the word error rate (WER), score a recogniser by slice (language, line, noise, accent), and choose between two settings that have almost the same WER.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The normalisation guide and why both texts go through it, from [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M02-L03-the-errors-that-matter/the-errors-that-matter-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/recordings/asr/whisper-tiny.jsonl": "6c6026f6b4971dd04e945beeb3d4a155eacc6ba54d77452dff27f5082c9f3104",
    "data/recordings/asr/whisper-small.jsonl": "67f27c0e6f5aa73d0e67cdb4b69c0e2d22c5f03a5dd74be14cbd8db0df17cb3f",
    "data/recordings/asr/vosk-small.jsonl": "ad050936f74c20db430f7668e4baa5e97ee84ce9cfdf5f0f8d98e8a8f024fe4d",
    "data/recordings/asr/whisper-base-prompt.jsonl": "2d259249892d3d86c0890052f547d62fd75d07a9f34bb6a0d59ab2fba075b14e",
    "data/recordings/asr/whispercpp-base.jsonl": "de4c4a6fb1e7ce5b7a1e191cddcc57af0e10fff9222eb11252be68491189e73d",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One call, one wrong number

Every call's labels have a field `important`: the words that a transcript must keep. Call C-0052 is a cancelled order on a mobile line with street noise. Run the next cell to load the calls and the recorded transcripts, and to print the important words of C-0052.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]

def recorded(method):
    """The recorded transcripts of one recogniser on the phone audio: {call_id: text}."""
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{method}.jsonl", encoding="utf-8")]
    return {r["call_id"]: r["text"] for r in rows if r["kind"] == "phone"}

tiny, small, vosk = recorded("whisper-tiny"), recorded("whisper-small"), recorded("vosk-small")
call = calls["C-0052"]
print(call["important"])

`cp.important_errors(important, model_text, language)` normalises the model text with the full guide and checks each item: an item is right only if all its words appear, in order, somewhere in the text. Run the next cell with the recorded Whisper tiny transcript of C-0052, first on the whole call, then on the caller's turn alone.

In [ ]:
hyp = tiny["C-0052"]
print("WER of the whole call:", round(cp.wer(cp.normalise(reference(call)), cp.normalise(hyp)), 3))
print("whole call:   ", cp.important_errors(call["important"], hyp, "en"))
caller_said = "Sure, the order number is 2091-17."   # Whisper tiny's words for the caller's turn
print("caller's turn:", cp.important_errors({"order": call["important"]["order"]}, caller_said, "en"))

> **Check.** The WER of the whole call is 0.087. On the whole call, `order` has `missed` 0 and `product` has `missed` 1 (Whisper tiny wrote "Cedar Rays better" for "Cedar raised bed"). On the caller's turn, `order` has `missed` 1: Whisper tiny wrote "2091-17" for "280917".

Why does the whole call keep the order number? The agent reads the number back ("That's order 280917"), and Whisper tiny wrote that part right. The check finds the number there. A phone menu has no agent who reads the number back: the caller's words are all it gets. Module 5 splits a call by speaker, so you can check the caller's words alone.

## 2. The important words on 60 test calls

Now the same count on every test call, for three recognisers.

In [ ]:
def important_table(texts):
    """Missed important items over the test calls: {kind: {"items": n, "missed": m}}."""
    rows = [cp.important_errors(c["important"], texts[c["call_id"]], c["language"]) for c in test]
    return cp.sum_important(rows)

def corpus_wer(texts, calls_):
    return cp.corpus_scores([(reference(c), texts[c["call_id"]], c["language"]) for c in calls_])["wer"]

for name, texts in [("Whisper tiny", tiny), ("Whisper small", small), ("Vosk small", vosk)]:
    t = important_table(texts)
    print(f"{name:14} WER {corpus_wer(texts, test):6.1%}  " +
          "  ".join(f"{k} {t[k]['missed']}/{t[k]['items']}" for k in ["order", "product", "date", "negation"]))

> **Check.** Order numbers missed: 21 of 52 (tiny), 8 of 52 (small), 37 of 52 (Vosk). Product names missed: 53, 41 and 51 of 60. The product names are spelled the Larkfield way ("GrowHub"), and any other spelling counts as a miss. Module 4 shows how a word list fixes most of them.

## 3. WER by slice

A slice is a group of calls that share one property: the language, the phone line, the noise or the accent of the caller's voice. One number for all 60 calls can hide a weak slice.

In [ ]:
def by_slice(texts, key):
    groups = {}
    for c in test:
        groups.setdefault(c[key], []).append(c)
    for value, items in sorted(groups.items()):
        print(f"   {value:10} {len(items):3} calls  WER {corpus_wer(texts, items):6.1%}")

for key in ["language", "line", "noise", "accent"]:
    print("Whisper small by", key)
    by_slice(small, key)

> **Check.** English 3.2% and French 15.1%; landline 8.6%, mobile 6.0%, VoIP 12.3%. French is the hard part here, not the noise. A slice with 7 calls (VoIP) moves a lot with each word, so treat small differences as noise.

## 4. Failure case: a check passes, and the meaning is gone

The negation check looks only for "not", "no", "don't" and "cannot". Whisper tiny turned "I do not want a refund, I want a new one." (call C-0056) into "I do not want to be feeling forward to new work."

In [ ]:
refund = calls["C-0056"]
sentence = "I do not want a refund, I want a new one."
print("Whisper tiny wrote:", "I do not want to be feeling forward to new work.")
print(cp.important_errors({"negation": [sentence]}, "I do not want to be feeling forward to new work.", "en"))
print("'refund' kept:", " refund " in " " + cp.normalise("I do not want to be feeling forward to new work.") + " ")

> **Check.** `negation` has `missed` 0: the check passes, because "not" is still there. But "refund" is gone, and nobody can tell what the caller wants. **Fix:** a check measures only what its rule looks for. Read a sample of the transcripts by hand, and add the words of the decision (here, "refund") to the important words when they matter.

## 5. Your turn: order numbers by line

> **Your turn.** For Whisper small, count the order numbers missed on each kind of line. Build a dictionary `orders_missed_by_line` from the line (`c["line"]`) to the number of missed order numbers on the test calls of that line. Use `cp.important_errors` with only the `order` items of each call.

> **Predict.** The VoIP line has the highest WER. Does it also miss the most order numbers?

In [ ]:
orders_missed_by_line = {}
for c in test:
    e = cp.important_errors({"order": c["important"]["order"]}, small[c["call_id"]], c["language"])
    orders_missed_by_line[c["line"]] = orders_missed_by_line.get(c["line"], 0) + e["order"]["missed"]
print(orders_missed_by_line)

In [ ]:
expect_hash('orders_missed_by_line', orders_missed_by_line, '48a07944c11e6903')

> **Check.** The landline misses the most order numbers, because most of the test calls (43 of 60) are landline calls. Divide by the number of order numbers on each line before you compare the lines: `speech score --method whisper-small --by line` prints both.

## 6. Two settings with almost the same WER

Module 4 compares many settings of Whisper. Here are two of them, both Whisper base: **A** gets a short list of Larkfield's words before it starts, and **B** is the same model run by another program, whisper.cpp.

In [ ]:
a, b = recorded("whisper-base-prompt"), recorded("whispercpp-base")
for name, texts in [("A: Whisper base, with a word list", a), ("B: Whisper base in whisper.cpp", b)]:
    t = important_table(texts)
    print(f"{name:34} WER {corpus_wer(texts, test):6.1%}  " +
          "  ".join(f"{k} {t[k]['missed']}/{t[k]['items']}" for k in ["order", "product", "date", "negation"]))

> **Check.** A: WER 15.2%, B: WER 15.3%. The difference is less than one point: noise. But A misses 18 product names against 45 for B, and 14 order numbers against 18. A also misses 3 negations against 1. Which one would you choose for Larkfield's search, and which one for an IVR that acts on what the caller says? Write your answer, then compare it with the knowledge check on the lesson page.

## Next

You can now count the errors that matter apart from the WER, and score a recogniser by slice. Module 3 looks at the parts of a call where nobody speaks. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/) has the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.